In [0]:

# ============================================================
# BRONZE - INGESTA DE PELICULAS DESDE AMAZON S3
# ============================================================

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_analytics"

# Ubicación externa configurada en Databricks
S3_RAW = "s3://proyecto-medallion-ventas-jleyvac-2026/raw/"

# Crear esquema si todavía no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Leer los tres CSV desde S3
df_movies = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(S3_RAW + "Movies.csv")
)

df_details = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(S3_RAW + "FilmDetails.csv")
)

df_posters = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(S3_RAW + "PosterPath.csv")
)

# Mostrar estructura y cantidad de registros
print("Movies:", df_movies.count())
print("FilmDetails:", df_details.count())
print("PosterPath:", df_posters.count())

display(df_movies.limit(5))
display(df_details.limit(5))
display(df_posters.limit(5))


Movies: 9718
FilmDetails: 9718
PosterPath: 9718


id,title,genres,language,user_score,runtime_hour,runtime_min,release_date,vote_count
1,The Shawshank Redemption,"Drama, Crime",en,8.7,2,22,1994-09-23,27070
2,The Godfather,"Drama, Crime",en,8.7,2,55,1972-03-14,20563
3,The Godfather Part II,"Drama, Crime",en,8.6,3,22,1974-12-20,12403
4,Schindler's List,"Drama, History, War",en,8.6,3,15,1993-12-15,15810
5,12 Angry Men,Drama,en,8.5,1,37,1957-04-10,8611


id,director,top_billed,budget_usd,revenue_usd
1,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton, William Sadler, Clancy Brown",25000000,28341469
2,Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan, Robert Duvall, Richard S. Castellano",6000000,245066411
3,Francis Ford Coppola,"Al Pacino, Robert Duvall, Diane Keaton, Robert De Niro, John Cazale",13000000,102600000
4,Steven Spielberg,"Liam Neeson, Ben Kingsley, Ralph Fiennes, Caroline Goodall, Jonathan Sagall",22000000,321365567
5,Sidney Lumet,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G. Marshall, Jack Klugman",397751,4360000


id,poster_path,backdrop_path
1,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,https://image.tmdb.org/t/p/w500/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg
2,https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,https://image.tmdb.org/t/p/w500/tmU7GeKVybMWFButWEGl2M4GeiP.jpg
3,https://image.tmdb.org/t/p/w500/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,https://image.tmdb.org/t/p/w500/kGzFbGhp99zva6oZODW5atUtnqi.jpg
4,https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,https://image.tmdb.org/t/p/w500/zb6fM1CX41D9rF9hdgclu0peUmy.jpg
5,https://image.tmdb.org/t/p/w500/ow3wq89wM8qd5X7hWKxiRfsFf9C.jpg,https://image.tmdb.org/t/p/w500/qqHQsStV6exghCM7zbObuYBiYxw.jpg


In [0]:

# ============================================================
# BRONZE - GUARDAR LOS DATASETS EN DELTA
# ============================================================

BRONZE = f"{CATALOG}.{SCHEMA}"

# Guardar cada CSV como tabla Delta Bronze
(
    df_movies.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.bronze_movies")
)

(
    df_details.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.bronze_film_details")
)

(
    df_posters.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.bronze_poster_paths")
)

print("Tablas Bronze creadas correctamente.")


Tablas Bronze creadas correctamente.


In [0]:

display(
    spark.sql(f"""
        SHOW TABLES IN {CATALOG}.{SCHEMA}
    """)
)


database,tableName,isTemporary
retail_analytics,bronze_film_details,false
retail_analytics,bronze_movies,false
retail_analytics,bronze_pedidos,false
retail_analytics,bronze_poster_paths,false
retail_analytics,bronze_productos,false
retail_analytics,gold_ventas,false
retail_analytics,silver_pedidos,false
retail_analytics,silver_productos,false
